# Практическая работа 10 (Тест)
## Тема: Параметрические критерии проверки статистических гипотез

### Настройка окружения

In [1070]:
import pandas as pd
import numpy as np
import scipy.stats as stats

### Вопрос 1
Предположим, что у нас есть следующие данные: $X = [12, 21, 33, 14, 17, 26]$. Мы хотим проверить нулевую гипотезу о том, что среднее значение равно 23. Чему в таком случае будет равно *t*-наблюдаемое? Выберите один верный ответ.

In [1071]:
x = np.array([12, 21, 33, 14, 17, 26])
mu_0 = 23

t_stat, _ = stats.ttest_1samp(x, popmean=mu_0)

round(t_stat, 2)

np.float64(-0.77)

### Вопрос 2
Какой будет критическая область *t*-статистики для двустороннего теста при уровне значимости 0.05 и 20 степенях свободы?

In [1072]:
alpha = 0.05
df = 20

t_crit = stats.t.ppf(1 - alpha / 2, df)

round(t_crit, 3)

np.float64(2.086)

*Ответ:* $(-\infty, -2.086] \cup [2.086, +\infty)$

### Вопрос 3
Изучалось влияние некоторого лекарственного препарата на артериальное давление до и после принятия этого препарата. Определите достоверность различия средних показателей на уровне значимости $\alpha=0.05$. Сделайте соответствующие выводы.

In [1073]:
before = np.array([250, 211, 205, 235, 225, 232, 211, 222, 218, 243, 219, 194, 208])
after = np.array([246, 211, 202, 233, 224, 229, 209, 222, 219, 243, 219, 194, 208])
alpha = 0.05
df = len(before) - 1

*Анализ по плану:*

1. Нулевая гипотеза: влияния лекарственного препарата на артериальное давление ***несущественное***.

2. Среднее значение разниц между давлением после и до приема лекарства, округленное до 2 знаков после запятой:

In [1074]:
diff = after - before
mean_diff = diff.mean()

round(mean_diff, 2)

np.float64(-1.08)

3. Значение дисперсии для разниц, округленное до 2 знаков после запятой:

In [1075]:
var_diff = diff.var(ddof=1)

round(var_diff, 2)

np.float64(2.41)

4. Наблюдаемое значение *t*-статистики, округленное до 2 знаков:

In [1076]:
t_stat, p_value = stats.ttest_rel(before, after)

round(t_stat, 2)

np.float64(2.5)

5. Число степеней свободы:

In [1077]:
df

12

6. Критическое значение критерия для $\alpha=0.05$ и найденного числа степеней свободы, округленное до 2 знаков после запятой:

In [1078]:
t_crit = stats.t.ppf(1 - alpha / 2, df)

round(t_crit, 2)

np.float64(2.18)

7. С использованием библиотеки `scipy.stats` рассчитайте значение *pvalue*, округленное до 2 знаков после запятой:

In [1079]:
round(p_value, 2)

np.float64(0.03)

*Вывод:* Нулевая гипотеза ***отвергается***.

### Вопрос 4
Файл [stress.json](./stress.json) содержит результаты изучения показателя стрессоустойчивости (в баллах) для двух профессий: учителя и врача, две группы по 25 и 20 человек соответственно. Для уровня значимости $\alpha=0.01$ необходимо проверить гипотезу о несущественности различий в средних значениях стрессоустойчивости у учителей и врачей. Сделать соответствующие выводы.

In [1080]:
import json

with open("stress.json", encoding="utf-8") as f:
    data = json.load(f)

teachers = np.array(data["teachers"])
n_teachers = len(teachers)

doctors = np.array(data["doctors"])
n_doctors = len(doctors)

*Анализ по плану:*

1. Нулевая гипотеза: в среднем учителя и врачи имеют ***одинаковую*** стрессоустойчивость.

2. Дисперсия стрессоустойчивости учителей, округленная до 2 знаков после запятой:

In [1081]:
var_teachers = teachers.var(ddof=1)

round(var_teachers, 2)

np.float64(12.08)

3. Дисперсия стрессоустойчивости врачей, округленная до 2 знаков после запятой:

In [1082]:
var_doctors = doctors.var(ddof=1)

round(var_doctors, 2)

np.float64(9.54)

4. Объединенная стандартная ошибка для выборок учителей и врачей, округленная до 2 знаков после запятой:

In [1083]:
ratio_teachers = var_teachers / n_teachers
ratio_doctors = var_doctors / n_doctors

se_combined = np.sqrt(ratio_teachers + ratio_doctors)

round(se_combined, 2)

np.float64(0.98)

5. Чему равна разность между средними значениями стрессоустойчивости учителей и врачей, округленная до 2 знаков после запятой:

In [1084]:
mean_teachers = teachers.mean()
mean_doctors = doctors.mean()

mean_combined = mean_teachers - mean_doctors

round(mean_combined, 2)

np.float64(-0.2)

*Проверка на соотношение дисперсий:*

In [1085]:
max_var = max(var_teachers, var_doctors)
min_var = min(var_teachers, var_doctors)

var_ratio = max_var / min_var

equal_var = var_ratio < 4

6. Наблюдаемое значение *t*-статистики, округленное до 2 знаков после запятой:

In [1086]:
t_stat, p_value = stats.ttest_ind(teachers, doctors, equal_var=equal_var)

round(t_stat, 2)

np.float64(-0.2)

7. Число степеней свободы:

In [1087]:
if equal_var:
    df = n_doctors + n_teachers - 2
else:
    # Поправка Уэлча - Саттертуэйта
    numerator = (ratio_teachers + ratio_doctors) ** 2
    denominator = (
        1 / (n_teachers - 1) * ratio_teachers**2
        + 1 / (n_doctors - 1) * ratio_doctors**2
    )

    df = numerator / denominator

df

43

8. Критическое значение критерия для $\alpha=0.01$ и найденного числа степеней свободы, округленное до 2 знаков после запятой:

In [1088]:
alpha = 0.01

t_crit = stats.t.ppf(1 - alpha / 2, df)

round(t_crit, 2)

np.float64(2.7)

9. С использованием библиотеки `scipy.stats` рассчитайте значение *pvalue*, округленное до 2 знаков после запятой:

In [1089]:
round(p_value, 2)

np.float64(0.84)

*Вывод:* Нулевая гипотеза ***принимается***.

### Вопрос 5
Набор данных [running.csv](./running.csv) содержит результаты опроса студентов некоторого британского вуза.

*Описание переменных:*

- **Athlete** - является ли студент спортсменом;
- **MileMinDur** - время, за которое студент пробегает милю.

Проверьте гипотезу на уровне значимости $\alpha=0.05$ о наличии разницы, за сколько секунд пробегают милю студенты-спортсмены и не спортсмены.

In [1090]:
survey = pd.read_csv("Running.csv")

survey.head(1)

,Athlete,MileMinDur
0,0,0:06:21


*Подготовка данных:*

1. Выразите значения столбца **MileMinDur** в секундах

In [1091]:
survey["MileSecDur"] = pd.to_timedelta(survey["MileMinDur"]).dt.total_seconds()

survey.head(1)

,Athlete,MileMinDur,MileSecDur
0,0,0:06:21,381.0


2. Разделите данные на группы

In [1092]:
athletes = survey[survey["Athlete"] == 1]["MileSecDur"]
n_athletes = len(athletes)

non_athletes = survey[survey["Athlete"] == 0]["MileSecDur"]
n_non_athletes = len(non_athletes)

*Анализ по плану:*

1. Нулевая гипотеза: студенты-спортсмены и не спортсмены в среднем пробегают милю за ***одно*** время.

2. Дисперсия времени бега для студентов-спортсменов, округленная до 2 знаков после запятой:

In [1093]:
var_athletes = athletes.var(ddof=1)

round(var_athletes, 2)

np.float64(2444.86)

3. Дисперсия времени бега для студентов не спортсменов, округленная до 2 знаков после запятой:

In [1094]:
var_non_athletes = non_athletes.var(ddof=1)

round(var_non_athletes, 2)

np.float64(14802.28)

4. Объединенная стандартная ошибка для выборок студентов-спортсменов и не спортсменов, округленная до 2 знаков после запятой:

In [1095]:
ratio_athletes = var_athletes / n_athletes
ratio_non_athletes = var_non_athletes / n_non_athletes

se_combined = np.sqrt(ratio_athletes + ratio_non_athletes)

round(se_combined, 2)

np.float64(8.96)

5. Чему равна разность между средними значениями времени бега студентов-спортсменов и не спортсменов, округленная до 2 знаков после запятой:

In [1096]:
mean_athletes = athletes.mean()
mean_non_athletes = non_athletes.mean()

mean_combined = mean_athletes - mean_non_athletes

round(mean_combined, 2)

np.float64(-134.79)

*Проверка на соотношение дисперсий:*

In [1097]:
max_var = max(var_athletes, var_non_athletes)
min_var = min(var_athletes, var_non_athletes)

var_ratio = max_var / min_var

equal_var = var_ratio < 4

6. Наблюдаемое значение *t*-критерия, округленное до 2 знаков после запятой:

In [1098]:
t_stat, p_value = stats.ttest_ind(non_athletes, athletes, equal_var=equal_var)

round(t_stat, 2)

np.float64(15.05)

7. Число степеней свободы с поправкой Уэлча, округленное вниз до ближайшего меньшего целого числа:

In [1099]:
if equal_var:
    df = n_doctors + n_teachers - 2
else:
    # Поправка Уэлча - Саттертуэйта
    numerator = (ratio_athletes + ratio_non_athletes) ** 2
    denominator = (
        1 / (n_athletes - 1) * ratio_athletes**2
        + 1 / (n_non_athletes - 1) * ratio_non_athletes**2
    )

    df = numerator / denominator

np.floor(df)

np.float64(315.0)

8. Критическое значение *t*-критерия для $\alpha=0.05$ и найденного числа степеней свободы, округленное до 2 знаков после запятой:

In [1100]:
alpha = 0.05

t_crit = stats.t.ppf(1 - alpha / 2, df)

round(t_crit, 2)

np.float64(1.97)

9. С использованием библиотеки `scipy.stats` рассчитайте значение *pvalue*, в ответе укажите порядок экспоненциальной формы записи числа:

In [1101]:
"{:.2e}".format(p_value)

'5.82e-39'

*Вывод:* Нулевая гипотеза ***отвергается***.